# Data Cleaning & Processing

In [121]:
import pandas as pd 
import numpy as np
import seaborn as sb
import matplotlib_inline as mp

I need to convert columns to their correct data-types.
Why? Because i wont be able to perform mathematical operations with non-numericals.

Pandas has a method that infers the correct type of data -> .convert_dtypes()
I don't want to do it manually like this: df['Age'] = pd.to_numeric(df['Age'])
Important Note: The method doesn't apply to datetime. -> Must do seperate operation.

In [122]:
df = pd.read_csv("messy_dataset.csv")
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.convert_dtypes()

I will drop duplicates early.
Dropping duplicates early means invalid data wont be included in mathematical operations later on.

It's smart to check if the duplicates aren't just rows that coniciedently share the same data.
.duplicated() checks for ID duplicates, so no two ID's can co-exist at the same time (must be unique) therefore i remove them.

In [123]:
print(f'Duplicates BEFORE removing: {df.duplicated(subset=['ID']).sum()}')
df = df.drop_duplicates(subset=['ID'])
print(f'Duplicates AFTER removing: {df.duplicated().sum()}')

Duplicates BEFORE removing: 500
Duplicates AFTER removing: 0


I can use skewness as the first step to determine if i use mean or median to replace empty cells.

If the data is right skewed (aka positive skewed) then most data points are on the left with few large values pushing the distribution to the right.
If the data is left skewed (aka negative skewed) then most data points are on the right with few large values pushing the distrubtion to the left.

The more appropriate average is median if the distribution is right-skewed or left-skewed as outliers will pull down or push the true value of the mean.
(Thresholds: -0.5 < skew  or skew > 0.5 )

No skewness / negligable skewness means i will use mean because no outliers are distorting the mean.
(skewness ≈ 0).

In [124]:
columns = ['Age','Sales','Profit','Discount']
for col in columns:
    skew = df[col].skew()

    if skew > 0.5:
        print(f'{col}: is left skewed, use median')

    elif skew < -0.5:
        print(f'{col}: is right skewed, use median')

    else:
        print(f'{col}: has no skew use mean')

Age: has no skew use mean
Sales: has no skew use mean
Profit: has no skew use mean
Discount: has no skew use mean


Before i use the mean i will check for outliers.  
Why? To see the mean is being distorted by outliers.  
(Using the IQR method (interquartile range method))  

In [125]:
columns = ['Age','Sales','Profit','Discount']
for col in columns:
    quartile1 = df[col].quantile(0.25)
    quartile3 = df[col].quantile(0.75)

    IQR =  quartile3 - quartile1

    lower_bound = quartile1 - (1.5 * IQR)
    upper_bound = quartile3 + (1.5 * IQR)

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"{col}: {len(outliers)} outliers")

Age: 0 outliers
Sales: 0 outliers
Profit: 0 outliers
Discount: 0 outliers


No outliers detected.  
It's safe to use the mean to the fill the empty cells.  
I will round to the closest whole number as the result may be a very long float numeric.  
(Pandas likely infered int for the columns not float so filling in anything but an int would raise an error)

For catergorial data like gender i can replace null values with either the mode or my own value.
I will just replace null values with 'Unknown'.

For category i will remove the rows.
Why? The data is useless now, a business or an individual can interpret nothing from sales figures if it's unknown what is being sold.

In [126]:
df[columns] = df[columns].fillna(df[columns].mean().round())
df['Gender'] = df['Gender'].fillna('Unknown')
df.dropna(subset=['Category'], inplace=True)

# Data Wrangling & Transformation

The data is clean.  
I can now perform some operations using formulas to get some useful information.   
Such as the following: 
- TOTAL SALES
- TOTAL PROFIT
- Profit Margin
- Profit after discounts
- Total revenue lost to discounts

Can also provide time-frames for the data averages i.e.: Annualy, monthly, weekly, daily.

.groupby() method will split the data into groups based on criteria (category).
.reset_index() method will provide the categorys an index rather than being the index.

Note: im unsure if sales is the sales revenue or the individual product count sold by the people.
I will assume its sales revenue

In [127]:
sales_summary = df.groupby('Category')['Sales'].sum().reset_index()

df['Profit Margin'] = (df['Profit'] / df['Sales']).round(decimals=2) 
df['Revenue Lost'] =  (df['Profit'] * df['Discount'] / 100).round(decimals=2)
df['Net Profit'] = (df['Profit'] - df['Revenue Lost']).round(decimals=2)
total_Revenue = df['Sales'].sum()
total_profit = df['Profit'].sum()
total_revenue_lost = df['Revenue Lost'].sum()


print(df)

        ID     Name  Age   Gender  Sales  Profit       Date     Category  \
0        1    David   25   Female    100      30 2023-01-01      Grocery   
1        2      Eve   27   Female    248      10 2023-01-01     Clothing   
2        3  Charlie   35    Other    248      30 2023-01-01  Electronics   
3        4      Eve   35  Unknown    248      10        NaT     Clothing   
4        5      Eve   27     Male    100      10 2023-02-15  Electronics   
...    ...      ...  ...      ...    ...     ...        ...          ...   
4992  4993      Eve   25   Female    400      10 2023-01-01  Electronics   
4994  4995      Bob   30     Male    248      10        NaT     Clothing   
4996  4997    David   30     Male    300      25        NaT    Furniture   
4998  4999    David   35     Male    248      25 2023-01-01      Grocery   
4999  5000      Eve   30     Male    100      40        NaT      Grocery   

      Discount  Profit Margin  Revenue Lost  Net Profit  
0           20            0.3

NOTE: SUCCESS CRITERIA WANTS LAMBDAS AND APPLY METHODS.